# Exp4 - Iterative GRPO on a fully open model stack

Trains a Llama-3.2-1B therapist to do Motivational Interviewing with **Group Relative Policy
Optimization**, against a simulated patient, with an oracle LLM grading validated MI
questionnaires as the reward. The run is **iterative**: every iteration regenerates its training
data from the *current* policy.

| Iteration `n` | does |
| --- | --- |
| 1. Generate | `pi_(n-1)` simulates 96 conversations against the patient, one per persona, into `conversations/<ARM>/model_iter_{n-1}/`. **Those conversations ARE the eval set for model state `n-1`** - there is no separate eval pass, which is why the folder is named after the policy that generated it. |
| 2. Extract | slice each conversation after every patient turn whose conversation-so-far has at least `MIN_CONV_LENGTH` utterances. |
| 3. Train | `GRPOTrainer` samples `G` completions per prompt, the reward function grades each one (through the K-turn look-ahead when `K > 0`), advantages are group-relative `(r - mean)/std` over the G siblings. |
| 4. Save | per-epoch checkpoints under `iteration_N/training/`, the final adapter at `iteration_N/adapter/` - **whose existence is the definition of "this iteration is done"**. |

After the loop, one generate-only pass produces `model_iter_{NUM_ITERATIONS}` so the final adapter
has matched eval data. `N` iterations therefore yield `N+1` conversation folders.

## What is different from Exp3: the stack is open, so the run costs $0 in API

The patient and the oracle are **open models served by one local vLLM OpenAI-compatible server**
(`plan_servers` dedupes by model id, so patient + oracle + judge on the same Gemma is ONE process,
one prefix cache, one weight copy). Exp3's binding constraint was the OpenAI bill, and on
2026-08-20 it stopped being theoretical - an organization spend cap killed two Colab sessions
outright. The only cost of this notebook is Colab GPU-hours.

**Exp3 and Exp4 scores are NOT comparable.** Different grader means a different axis. Compare
within Exp4 only.

## The cell order is a contract, not a style

1. flat globals (section 1)
2. runtime detect + auth (section 2)
3. **`serve_roles()` - section 3, BEFORE any torch import.** `gpu_memory_utilization` is a
   *pre-allocation*, not a growing ceiling: vLLM grabs its share of the card at startup and never
   gives it back. The server must carve out its small fixed reservation first, because training
   memory is the spiky side.
4. the oracle-sanity gate (section 4) - an open grader can fail *silently*, and this is the only
   thing that catches it
5. `import trl` **then** torch (section 5). On the local Blackwell card (sm_120) the other order
   segfaults at CUDA init: exit 139, no traceback, nothing to catch. Colab is unaffected.
6. build the config bundle (section 6) - **`EXPERIMENT_NAME` is computed here, never typed**
7. the visible orchestration loop (section 8)

## Where the code lives

| Module | Responsibility |
| --- | --- |
| `core/` | Everything both methods share: config freezing + path shapes, conversations, K-turn look-ahead, oracle scoring, the TRL reward callable, the EDA recorder, timing, TensorBoard. |
| `grpo/grpo_trainer.py` | The pieces ONE iteration is composed from. There is deliberately no `run_iterative_training` in it: Exp3 had one, it duplicated the notebook loop, and the two drifted until calling the wrapper silently ran a different experiment. |
| `tools/vllm_serve.py` | The only module that knows there is a subprocess at all. |
| `tools/oracle_sanity.py` | The gate in section 4. |

## Resume

Re-run this notebook top-to-bottom. Completed iterations are skipped (`resolve_start_state` reads
the adapters on disk), conversations already written are reloaded rather than regenerated, a
crashed iteration resumes from its latest *valid* HF checkpoint, and the EDA recorder is restored
from the snapshot stored inside that checkpoint. Nothing here needs a flag flipped to resume.


---
## 0. Environment (self-installing, guarded)

Just run it. On a **fresh** Colab runtime it installs the stack and asks you to restart; on a
runtime that is already correct it prints one line and moves on. Nothing to uncomment.

It decides by reading installed package **metadata** (never importing anything — this cell must
not pull torch in), and it will not act in three cases: the environment already matches, you set
`AUTO_INSTALL = False`, or you are **not on Colab** — where installing would write vLLM and a CUDA
torch straight into the repo's `.venv`.

**Order matters when it does install.** vLLM brings its own torch wheel, so it goes **first** and
the pinned training stack is layered on top; installing vLLM last would silently replace the torch
that stack was resolved against. vLLM itself is unpinned (no validated version in
`requirements.txt`), but the cell warns if the installed build is older than **0.19.1**, the floor
for Gemma 4 support.

**Restart after a real install.** Pip replaces files on disk; a kernel that already imported the
old modules keeps holding them. Re-running this cell after the restart just prints the OK line.

In [ ]:
# =============================================================================
#  CELL 0 -- environment. This cell RUNS ITSELF; there is nothing to uncomment.
# =============================================================================
# GUARDED, not commented out. Three things it refuses to do:
#
#   * reinstall on a runtime that is already correct -- minutes of churn, and
#     re-installing vLLM re-resolves torch underneath a stack that was fine;
#   * install anything OUTSIDE Colab -- this notebook is importable locally for
#     smoke tests, and a vLLM/CUDA install into the repo's .venv is not
#     something to do by accident;
#   * install in the wrong order -- vLLM brings its own torch wheel, so it goes
#     FIRST and the pinned stack is layered on top. Installing vLLM last
#     silently replaces the torch the training stack was resolved against.
#
# So: a fresh runtime installs once and asks for a restart; every later run
# prints one line and moves on.
# ⚠ Byte-identical to the GRPO notebook's cell 0 on purpose -- the two install
# cells drifting apart is exactly how one method ends up on a different stack.
AUTO_INSTALL = True        # False -> report what is missing, change nothing

import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

IS_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))

# The repo's validated set (requirements.txt). torch is deliberately NOT pinned
# here: Colab ships a CUDA build and vLLM pins its own on top of it.
PINNED = {
    "accelerate": "1.13.0",
    "datasets": "4.8.5",
    "huggingface_hub": "1.14.0",
    "numpy": "2.4.4",
    "openai": "2.36.0",
    "pandas": "3.0.3",
    "peft": "0.19.1",
    "safetensors": "0.7.0",
    "tensorboard": "2.20.0",
    "transformers": "5.8.1",
    "trl": "1.4.0",
}
# vLLM is deliberately UNPINNED until a build has passed `smoke.py roles` on Colab
# (requirements.txt carries no validated version). Once one has, put its exact
# version in PINNED_VLLM -- None means "latest that clears VLLM_MIN". Gemma 4
# needs at least VLLM_MIN, per the official vLLM Gemma 4 recipe; an older build
# is treated as work to do, not as a warning to act on by hand: it cannot serve
# the grader at all, so section 3 would fail either way.
PINNED_VLLM = None          # e.g. "0.19.1" once a build passes smoke.py roles
VLLM_MIN = (0, 19, 1)


def _v(pkg):
    """Installed version string, or None. Reads package METADATA -- imports nothing.

    That matters here: this cell must not pull torch in, both for the sm_120
    import-order contract and because section 3 has to start vLLM first.
    """
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None


def _older_than(ver, minimum):
    """True iff *ver* parses as a release older than *minimum*. Unparseable -> False."""
    try:
        return tuple(int(x) for x in ver.split("+")[0].split(".")[:3]) < minimum
    except (AttributeError, ValueError):
        return False           # a dev/nightly build -- do not cry wolf


_off_pin = {p: (_v(p), want) for p, want in PINNED.items() if _v(p) != want}
_vllm = _v("vllm")
_vllm_bad = (_vllm is None or _older_than(_vllm, VLLM_MIN)
             or (PINNED_VLLM is not None and _vllm != PINNED_VLLM))
# Colab pre-bakes torchao < 0.16.0, and peft 0.19.1 does not merely ignore it:
# get_peft_model's LoRA dispatcher calls dispatch_torchao, which RAISES against
# that version, so attaching the adapter fails outright. Nothing here uses it.
_torchao = _v("torchao")
_work = bool(_off_pin) or _vllm_bad or _torchao is not None


def _report():
    for _p, (_have, _want) in sorted(_off_pin.items()):
        print(f"  {_p}: have {_have}, want {_want}")
    if _vllm is None:
        print("  vllm: not installed")
    elif _older_than(_vllm, VLLM_MIN):
        print(f"  vllm {_vllm}: older than {'.'.join(map(str, VLLM_MIN))}, the Gemma 4 floor")
    elif PINNED_VLLM is not None and _vllm != PINNED_VLLM:
        print(f"  vllm: have {_vllm}, want {PINNED_VLLM} (PINNED_VLLM)")
    if _torchao is not None:
        print(f"  torchao {_torchao}: installed, and peft 0.19.1 raises against it")


if not _work:
    print(f"environment OK -- {len(PINNED)} pinned packages match, "
          f"vllm {_vllm}, torchao absent")
elif not AUTO_INSTALL:
    print("AUTO_INSTALL=False -- nothing was changed. Outstanding:")
    _report()
elif not IS_COLAB:
    print("NOT Colab -- refusing to install (this would write into the local .venv).")
    print("Outstanding, for reference:")
    _report()
else:
    def _pip(*args):
        print(f"  $ pip {' '.join(args)}")
        subprocess.check_call([sys.executable, "-m", "pip", *args])

    if _vllm_bad:
        print(f"{'installing' if _vllm is None else f'upgrading vLLM {_vllm} ->'} vLLM FIRST "
              f"(it pins its own torch) -- takes several minutes")
        _pip("install", "-q", "-U", "vllm" if PINNED_VLLM is None else f"vllm=={PINNED_VLLM}")
    if _off_pin or _vllm_bad:
        # The FULL list, not just the off-pin subset: a fresh vLLM may have moved
        # numpy/transformers underneath it, so re-assert the whole validated set.
        print("pinning the training stack on top of vLLM's torch")
        _pip("install", "-q", *[f"{p}=={v}" for p, v in PINNED.items()])
    if _torchao is not None:
        print(f"removing torchao {_torchao} (peft 0.19.1 raises inside its LoRA dispatcher)")
        _pip("uninstall", "-y", "-q", "torchao")

    # Gate 1: the resolved set is CONSISTENT. `pip check` lists every broken requirement
    # (a vLLM that moved numpy or transformers underneath the pins, say). A non-zero exit is
    # a stack that would fail later, somewhere less legible, so it fails HERE.
    print("  $ pip check")
    _check = subprocess.run([sys.executable, "-m", "pip", "check"],
                            capture_output=True, text=True)
    print(_check.stdout.strip() or "  pip check: no broken requirements")
    if _check.returncode != 0:
        raise RuntimeError(f"pip check failed after the install:\n{_check.stdout}{_check.stderr}")

    # Gate 2: vLLM IMPORTS -- probed in a SUBPROCESS. Never import vllm (or torch) in THIS
    # kernel: section 3's server must claim its pre-allocation before torch initialises CUDA
    # here, and section 5's trl-before-torch order must survive on the local sm_120 card.
    _probe = subprocess.run([sys.executable, "-c", "import vllm, sys; print(vllm.__version__)"],
                            capture_output=True, text=True)
    if _probe.returncode != 0:
        raise RuntimeError("vLLM is installed but does not import in a fresh interpreter:\n"
                           + _probe.stderr[-2000:])
    print(f"  vllm imports in a fresh interpreter: {_probe.stdout.strip()}")

    print("\n" + "=" * 74)
    print("  RESTART THE RUNTIME NOW  (Runtime -> Restart session), then run the mount")
    print("  cell and this cell again -- it will print one line and skip. Installing")
    print("  over modules the kernel has already imported leaves it holding the old ones.")
    print("=" * 74)
    # Raised on purpose so Run-all STOPS here: every cell below would otherwise run on the
    # stale, half-imported stack and fail somewhere that looks like a training bug.
    raise RuntimeError("Runtime restart required: Runtime > Restart session, then re-run the "
                       "mount cell and this cell")

# bitsandbytes is only needed for USE_4BIT=True, which Exp4 never runs (4-bit induced ~30x
# more phrase-loop degeneration on this same base model in Exp2, which moved the whole score
# axis). If you ever flip that toggle: pip install bitsandbytes==0.49.2

---
## 1. Configuration - the only cell in this notebook where a number is typed

Everything downstream is handed an already-decided value: `core.config` freezes these flat globals
into typed dataclasses, and no other module re-reads a global or joins a path.

**`EXPERIMENT_NAME` is NOT here.** It is *computed* in section 6 from the rubric, K, MCL, G and the
oracle and patient model ids. Exp3 typed it by hand as an f-string that mentioned `LA{K}` and
`MCL{N}` but not the roles - so changing `ORACLE_MODEL_ID` wrote a differently-rewarded policy into
the default arm's folder, where the resume-by-skipping scorer then reported "already scored"
against another arm's numbers. A name nobody is allowed to type cannot disagree with the config it
describes.

**What the arm name does NOT encode** is everything else: the learning rate, every temperature,
`NUM_ITERATIONS`, the look-ahead sub-batch. Change one of those and the folder name is
byte-identical. `run_metadata.json` (written in section 6, with an append-only history log beside
it) is the only record that can tell two runs of the "same" arm apart.


In [ ]:
# =============================================================================
#  CELL 1 -- flat globals.  EXPERIMENT_NAME IS NOT HERE; it is computed (sec. 6).
# =============================================================================

# --- Roles: who plays the patient, who grades --------------------------------
# The patient defines the TASK and the oracle defines the TRAINING TARGET, so
# swapping either makes arms incomparable -- which is why BOTH tags are encoded
# in the arm name, unconditionally. The judge grades after the fact and is
# re-runnable, so it partitions the score lake (judge=<tag>/) instead and does
# not appear in the name.
# All three default to one open model. plan_servers dedupes by model id, so this
# is ONE vLLM process serving three roles that differ only in sampling params --
# three servers would triple the weight memory and split the prefix cache, and
# prefix caching is exactly what makes the rubric-first oracle prompt cheap.
#
# MODEL CHOICE (both ungated, Apache 2.0; sizes read off the HF API 2026-08-26):
#   google/gemma-4-E4B-it  7.996B params, 14.89 GiB bf16  <- default: the grader IS the
#                                                            instrument; better odds on the
#                                                            sanity gate's rank-agreement bar
#   google/gemma-4-E2B-it  5.123B params,  9.54 GiB bf16  <- fallback if E4B is too slow/tight;
#                                                            roles.DEFAULT_SERVE_UTIL pins it
#                                                            to 0.35 (section 3 asserts it)
# Switching model = edit the three ids; the arm name picks up the new tag automatically. Run the
# FULL oracle_sanity against BOTH and choose by Spearman + spread, not by size.
ORACLE_PROVIDER  = "openai_compat"          # openai_compat | openai | anthropic
ORACLE_MODEL_ID  = "google/gemma-4-E4B-it"
PATIENT_PROVIDER = "openai_compat"
PATIENT_MODEL_ID = "google/gemma-4-E4B-it"
JUDGE_PROVIDER   = "openai_compat"
JUDGE_MODEL_ID   = "google/gemma-4-E4B-it"

# Gemma 4 ships configurable thinking modes, and thinking must be OFF for the
# oracle and the patient: a reasoning preamble in front of a schema-constrained
# response is a good way to lose the schema, and it costs latency on every one of
# the ~10k oracle calls an iteration makes.
# The `enable_thinking` key roles.thinking_off_extra_body() sends matches the
# official vLLM Gemma 4 recipe AND the model card (checked 2026-08-26), and Gemma 4
# ships with thinking OFF by default -- so this is belt and braces. `tools/smoke.py
# roles` still verifies no thinking tokens appear on the wire, because a wrong
# chat_template_kwargs key would fail SILENTLY (an unknown Jinja name is just an
# unused variable).
DISABLE_THINKING = True

# Per-ATTEMPT timeouts x MANY retries -- never a long total budget. Exp3's patient
# call had no timeout at all, so the openai SDK's 600 s default let one hung
# look-ahead call add ten minutes to an optimizer step. Worse, exhausting a long
# budget FREEZES a simulation, and under scale_rewards="group" one frozen sim
# shifts the mean AND the std of its whole group of 8: the damage is not confined
# to the sample that failed.
# NOTE the local vLLM server QUEUES requests and queue wait counts against the
# attempt, so the per-attempt bound carries headroom (90 s) rather than being
# vendor-API tight. Matched to the PTO notebook -- a timeout asymmetry between
# methods would be a method-confounded freeze probability.
PATIENT_REQUEST_TIMEOUT = 90.0    # one patient utterance
PATIENT_MAX_RETRIES     = 8
# ORACLE_REQUEST_TIMEOUT / ORACLE_MAX_RETRIES are read TWICE on purpose: once as
# the socket timeout of the oracle's client, once as OracleConfig's coroutine
# bound. Keep them equal -- a coroutine bound below the socket bound makes the
# retry count the only real budget.
ORACLE_REQUEST_TIMEOUT  = 120.0   # a scoring call emits a whole JSON rubric
ORACLE_MAX_RETRIES      = 3

# --- Serving: the one vLLM process -------------------------------------------
# gpu_memory_utilization is a PRE-ALLOCATION, not a ceiling that grows on demand.
# Sharing the card with a live trainer wants it as low as the weights allow and the
# server started FIRST (section 3), because training memory is the spiky side:
# GRPO's 128-completion generate plus the look-ahead rollout.
# Show the arithmetic (project rule). Target: the Colab A100 80 GB (2026-09-02).
#   server:  0.50 x 80 = 40 GiB = 14.89 GiB E4B weights + ~22 GiB KV pool + vLLM
#            overhead (E2B: 0.35 x 80 = 28 GiB = 9.54 + ~16 KV, per roles.DEFAULT_SERVE_UTIL).
#            The KV pool is what lets 96 patient calls + 128 oracle calls overlap.
#   trainer: the other ~40 GiB minus ~1 GiB for two CUDA contexts = ~38 GiB of room, of which
#              2.5   policy (1B bf16 + LoRA + Adam on the adapter)
#            + 8.8   the ONE 128-completion generate per optimizer step
#                    (128 x ~2.2k tokens x 32 KiB/token of KV on this 1B)
#            + 4.4   the look-ahead generate at LOOKAHEAD_SUB_BATCH_SIZE=64 (K=5 only)
#            + 4.0   the 16-completion loss forward WITH gradient checkpointing
#                    (16 x 200 x 128k-vocab fp32 logits = 1.6 GiB, x2 old/ref, + activations)
#            = 19.7 GiB envelope (a PLAN number, unmeasured; the same sum smoke.py vram
#            pins as TRAINER_ENVELOPE_GIB). It gets MEASURED: every phase writes its
#            peak_reserved_gib_* into iteration_metadata.json (run QUICK_TEST first).
#   escape hatches, in order: LOOKAHEAD_SUB_BATCH_SIZE (auto-halves on OOM, sticky),
#            CONVERSATION_BATCH_SIZE, then E4B -> E2B (a science change: new arm name).
#            NEVER VLLM_MAX_MODEL_LEN (below) and never VLLM_GPU_MEM_UTIL below the
#            weights + a usable KV pool.
# On a 40 GB card the same 0.50 leaves the server only a ~5 GiB KV pool and the
# trainer 40 - 20 - 1 = ~19 GiB against the 19.7 GiB envelope: headroom ~= 0, so the
# fallback needs the escape hatches above from the start.
# (The old 0.25 default assumed "~3 GB" of Gemma weights; the real checkpoints are
# 3-5x that, and at 0.25 the E4B server cannot even hold its weights.) Verify the
# measured weights line section 3 prints before trusting any of this.
VLLM_PORT          = 8000
# ONE owner for this fraction: roles.DEFAULT_SERVE_UTIL (E4B 0.50 / E2B 0.35, sized from the
# measured bf16 checkpoints; smoke.py vram pins it). Section 3 asserts this literal equals
# roles.default_serve_util(<served model>) -- change the table, not one notebook.
VLLM_GPU_MEM_UTIL  = 0.50
# WARNING: 16384 is NOT negotiable downward as a memory escape hatch. Measured on
# the 192 real Exp3 PTO_LA0 transcripts, full oracle prompts (rubric + transcript)
# run to 9,319 tokens for Q1 and 10,042 for Q2; at 8192 that is 1.0% / 2.1% of
# conversations that cannot be scored AT ALL. Those are the LONGEST conversations,
# and session length varies by arm and by K -- so the dropout would be
# arm-dependent, a silent bias on the headline metric rather than an error (an
# unscoreable conversation is simply absent). The memory cost is near zero: the KV
# pool is sized by gpu_memory_utilization, and this caps ONE sequence rather than
# multiplying the pool. Give memory back via VLLM_GPU_MEM_UTIL instead.
VLLM_MAX_MODEL_LEN = 16384
VLLM_DTYPE         = "bfloat16"
VLLM_EXTRA_ARGS    = ()      # e.g. ("--enable-prefix-caching",) on older vLLM
# "" -> /content/vllm_logs on Colab, ./_vllm_logs locally (resolved in section 3).
# NEVER the Drive mount: the server's merged stdout streams through this path for
# the whole arm, and a wedged FUSE write on it can stall the server itself.
VLLM_LOG_DIR       = ""

# --- Reward: what the oracle grades ------------------------------------------
# Q1 + Q2 only, matching the ICLR look-ahead paper. The reward is the UNWEIGHTED
# mean across questionnaires, so Q1 (5 items) and Q2 (17 items) carry equal
# weight. If ANY single questionnaire fails validation the candidate's score is
# None -- "not graded", never "graded badly", because otherwise the reward's
# DEFINITION would depend on which calls happened to fail. That None never reaches
# TRL: the pinned trl 1.4.0 maps it to NaN and nansums it to 0.0 (i.e. optimises it
# as the worst possible completion and re-scales its seven siblings), so
# core.reward.rewards_for_trl substitutes the candidate's GROUP MEAN first --
# advantage ~0, group mean unchanged -- and records it as `reward_used` in the EDA.
QUESTIONNAIRE_IDS        = [1, 2]     # -> QTAG "Q1Q2" in the arm name
EVAL_TEMPERATURE         = 0.0        # >0 makes the grader a random variable
# Keep ORACLE_MAX_TOKENS equal to the sanity gate's budget in section 4. A local
# model that opens with a preamble can fit the JSON at 512 and clip it at 256,
# which surfaces only as retries and missingness once training is under way.
ORACLE_MAX_TOKENS        = 256
ORACLE_MAX_CONCURRENCY   = 64
# The reward function RAISES when a batch's oracle success rate falls below this.
# Training on a biased subset is worse than stopping: the calls that fail are the
# hard ones, so the survivors are not a random sample.
ORACLE_MIN_SUCCESS_RATIO = 0.5
# False omits strict:true from the json_schema response format, for a vLLM build
# that 400s on the key. A grader that only passes with this False is being held to
# the schema by the SERVER rather than following the rubric. Recorded in
# run_metadata.json (config.oracle.openai_compat_strict).
ORACLE_SCHEMA_STRICT     = True

# --- Look-ahead: the lever this experiment exists to measure -----------------
# K is purely about WHAT CONTEXT the oracle scores, never about the loss: it is
# the number of extra simulated turns appended after each candidate completion
# before the oracle is queried. K=0 scores "did this opening look good?", K>0
# scores "did it lead somewhere good under the current policy?".
LOOKAHEAD_K                = 0        # 0 | 5  -- encoded in the arm name as LA{K}
LOOKAHEAD_TEMP_THERAPIST   = 0.9
LOOKAHEAD_TEMP_PATIENT     = 0.7
LOOKAHEAD_MAX_TOKENS       = 200
LOOKAHEAD_MAX_INPUT_TOKENS = 2048
# Therapist generates per look-ahead turn are chunked at this width and the chunk
# is HALVED, stickily, on OOM. It is in no EXPERIMENT_NAME, so a halving leaves no
# trace except run_metadata.json + iteration_metadata.json -- and per-iteration
# wall-clock is only comparable between iterations that ran at the same value.
# 64 against 128 completions per optimizer step means two chunks per simulated turn.
LOOKAHEAD_SUB_BATCH_SIZE   = 64

# --- The loop ----------------------------------------------------------------
# NUM_ITERATIONS is NOT in the arm name, so changing it never forks the folder.
# Every completed iteration is independently usable and iteration n's step 1
# generates model_iter_{n-1}, so stopping early only ever costs the FINAL
# adapter's eval conversations -- which the post-loop pass in section 9 refills.
NUM_ITERATIONS       = 10
# 1 on purpose, matched across BOTH methods -- and 1 is not the same thing in the
# two trainers, which is exactly why 2 was wrong: a GRPO epoch re-SAMPLES G fresh
# completions per prompt and re-grades them (2 epochs = twice the reward-side
# work, on partially-updated weights), while a DPO epoch re-treads the SAME fixed
# pairs. epochs=1 makes "one pass over data produced by this iteration's policy"
# true for both methods; raise NUM_ITERATIONS, not this, for more updates.
EPOCHS_PER_ITERATION = 1
SEED                 = 42
# Audit only: Exp4 has NO mode_tag path level (Exp3 had full|quicktest).
# WARNING: a smoke run that keeps the same rubric, K, MCL, G and role models as a
# real arm resolves to the SAME folder and WILL pollute it. When smoke-testing an
# arm you intend to run for real, change something the name encodes (K, or G), or
# point DATA_ROOT at a scratch directory. (The EDA disambiguates display labels on
# collision and can pin an arm by experiment_name, so a mini-arm can never
# silently merge into the real arm's figures.)
RUN_MODE   = "full"
QUICK_TEST = False   # overrides at the BOTTOM of this cell; drops G to 4 -> a
                     # disjoint _G4_ folder, so a smoke run cannot pollute the real arm

# --- Conversations -----------------------------------------------------------
NUM_CONVERSATIONS_PER_ITER = 96       # one per V3 patient persona
# ADDITIONAL utterances after the scripted therapist opener (one loop step = one
# utterance, therapist + patient alternating) -> max 50 utterances per conversation
# at 49. Same number as Exp3; the off-by-one is documented, not changed.
NUM_UTTERANCES_FOR_DATA    = 49
# MCL: drop training slices whose conversation-so-far has fewer than this many
# utterances. This is the response to the reward-faithfulness finding -- a partial
# cut is what the training reward grades while the experiment evaluates whole
# conversations, and pairwise rank agreement between the two is barely above
# chance at 2 utterances and only clears 0.8 near 10. Encoded in the arm name.
MIN_CONV_LENGTH            = 12
# WARNING: on the local 12 GB card this is a SAFETY setting, not a throughput knob
# -- weights ~2.6 GB plus ~1.1 GB per concurrent conversation, and an over-budget
# VRAM request there REBOOTS THE MACHINE instead of raising OutOfMemoryError. On
# the A100 a large batch is what amortises the patient round-trip across the batch.
# OOM on a batch halves it STICKILY for the rest of the pass; a pass that still
# cannot complete every persona RAISES rather than train on a biased subset.
CONVERSATION_BATCH_SIZE    = 64
TEMPERATURE_THERAPIST_GEN  = 0.9
TEMPERATURE_PATIENT        = 0.7
MAX_TOKENS_PER_RESPONSE    = 200
# Both budgets are BOS-INCLUSIVE (the length of core.policy.prompt_token_ids), and an
# over-budget conversation drops its OLDEST turns WHOLE and keeps the system prompt
# (core.policy.build_prompt) -- never token-truncated, so the policy always sees
# the session start (Exp3 left-truncated at the token level past ~utterance 24 and
# lost the system prompt). KEEP THE TWO EQUAL: the SAME function builds the
# serve-time prompt and the training prompt, so at equal budgets the text TRL
# trains on is byte-identical to what the policy generated from; validate_config
# warns when they differ. The realised truncation rate is printed per batch
# (`trunc <n>/<B>`) and recorded per phase in iteration_metadata.json.
THERAPIST_MAX_INPUT_TOKENS = 2048     # serve time: conversation loop + look-ahead
MAX_ALLOWED_PROMPT_LENGTH  = 2048     # training prompt cap; == the line above
PATIENT_CONCURRENCY        = 96
MAX_GEN_RETRIES_WITHOUT_PROGRESS = 3
# "auto" resolves per therapist (core.config.resolve_stop_strings). BASE model:
# the ChatML pair -- there <|im_end|>/<|im_start|> are ordinary BPE pieces, not
# special tokens, and without string stops the 1B happily writes BOTH speakers
# to the token cap (polluting the transcript the oracle grades and, because GRPO
# credits every sampled token, training toward 200-token rambles). INSTRUCT
# model: EMPTY -- its native template ends every turn on the special <|eot_id|>,
# so stopping is exact token-id matching at zero per-call cost.
STOP_STRINGS         = "auto"
GEN_VERBOSE          = True
GEN_VERBOSE_DETAILED = False

# --- GRPO ---------------------------------------------------------------------
# TRAIN_BATCH_SIZE counts COMPLETIONS, not prompts, and it is the per-FORWARD shape:
# trl runs the with-grad loss forward over the whole per-device micro-batch in ONE
# pass (no chunking, no OOM fallback), so this is the VRAM lever. Exp3 measured
# 64 with checkpointing OFF at ~67 GB on an A100-80GB that had the whole card;
# here ~38 GiB remain beside the vLLM server, hence 16 x 8 + checkpointing ON.
# The optimizer step is UNCHANGED: generation_batch_size = 16 x 8 = 128 completions
# = (16 x 8) / G=8 = 16 unique prompts per step -- matched to PTO's 16 preference
# pairs so the two methods take comparable-sized steps -- and TRL still issues ONE
# generate() of all 128 per step (steps_per_generation defaults to gas), buffered
# and served to the 8 micro-batches of 16. Read prompts/step off
# cfg.prompts_per_step, do not recompute it.
# gas is gradient-scale-neutral on the pinned trl 1.4.0 (it bypasses transformers'
# training_step scaling via a non-None compute_loss_func sentinel and divides the
# loss exactly once itself, _compute_loss ~:2568-2570). The old "1/gas^2" warning
# was Exp3's earlier stack -- re-verify on any trl bump.
NUM_GENERATIONS             = 8       # G, matched to PTO's M
TRAIN_BATCH_SIZE            = 16      # completions per device = per-forward shape (VRAM)
EVAL_BATCH_SIZE             = 16
GRADIENT_ACCUMULATION_STEPS = 8       # -> 128 completions -> 16 prompts / optimizer step
MAX_COMPLETION_LENGTH       = 200
GRPO_BETA                   = 0.01    # KL against the ITERATION-START adapter
GRPO_TEMPERATURE            = 1.2
GRPO_LOSS_TYPE              = "grpo"
GRPO_INNER_ITERATIONS       = 1       # PPO inner iterations (mu), NOT the arm loop
LEARNING_RATE               = 1e-5
WARMUP_STEPS_RATIO          = 0.01
# Held-out CONVERSATIONS, not samples: every slice of one conversation shares its
# opening turns, so a sample-level split would measure memorisation. Eval is not
# free -- TRL samples G completions for every eval prompt and grades all of them.
EVAL_SPLIT_RATIO            = 0.05
# ON: at 16 completions x ~2.2k tokens the activations, not the logits, are the
# spike; non-reentrant (use_reentrant=False) so it composes with PEFT's frozen base,
# and trl enables the PEFT input grads itself when this is set.
GRADIENT_CHECKPOINTING      = True

# --- Policy + LoRA ------------------------------------------------------------
# The therapist IS encoded in the arm name (_Th{tag}), so the two variants can
# never share a folder:
#   meta-llama/Llama-3.2-1B-Instruct -> _ThL1Bi (default): native Llama-3 chat
#       template; turns end on the single special token <|eot_id|>, so stopping
#       is token-id-exact and STOP_STRINGS="auto" resolves to () -- the ChatML
#       self-play failure class does not exist here.
#   meta-llama/Llama-3.2-1B          -> _ThL1B: the template-less base; the
#       hand-written ChatML template is installed and the ChatML stop strings
#       are required (what "auto" resolves to there).
# run_metadata.json records the exact snapshot id (the tag names a family).
BASE_MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
TOKENIZER_ID  = ""        # "" -> BASE_MODEL_ID
# Exp4 never runs 4-bit. Exp2 generated in 4-bit NF4 and Exp3 in bf16 on the SAME
# base model, and 4-bit induced ~30x more phrase-loop degeneration (~9.5% vs ~0.3%
# of therapist turns running to the cap as repeated spam), which the oracle floors
# -- moving the whole score axis. A 1B in bf16 is ~2.5 GB: nothing to buy, a
# comparability hazard to lose.
USE_4BIT      = False
LORA_R        = 16
LORA_ALPHA    = 16
# 0.0, MATCHED with PTO, and the value is not the point -- dropout is OFF in both
# trainers: DPOConfig.disable_dropout defaults True, GRPOConfig's defaults False,
# and Exp4 previously set neither, so 0.05 was real in GRPO only. A reference-model
# method should not compare logps under stochastic dropout, so grpo_trainer passes
# disable_dropout=True explicitly (recorded in run_metadata.json) and any non-zero
# value here would be inert -- the config should say what the run does.
LORA_DROPOUT  = 0.0
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "up_proj", "down_proj", "gate_proj"]

# --- Logging, checkpointing, capture -----------------------------------------
REPORT_TO     = ["tensorboard"]   # Exp4 is TensorBoard-only; nothing creates a W&B run
LOGGING_STEPS = 1
SAVE_STRATEGY = "steps"
SAVE_STEPS    = 10
# >= 2 on purpose: a process killed DURING a checkpoint write leaves the newest one
# half-written, and get_latest_valid_hf_checkpoint has to have something to walk
# back to or the whole iteration is lost.
SAVE_TOTAL_LIMIT = 2
# rich prints NUM_COMPLETIONS_TO_PRINT (prompt, completion, reward) rows into the
# cell every logging step; None would print all 128 per step. The per-step parquet
# capture under training/completions/ is unaffected by the print count.
LOG_COMPLETIONS          = True
NUM_COMPLETIONS_TO_PRINT = 4
# iteration_N/eda/generations.jsonl: one row per prompt-group with all G candidates
# nested (completion, score, per-questionnaire sub-scores, the look-ahead tail the
# oracle actually scored). SAVE_LOOKAHEAD_TRANSCRIPTS is the dominant size lever.
SAVE_EDA_GENERATIONS       = True
SAVE_LOOKAHEAD_TRANSCRIPTS = True
# Opt-in run-level continuous TB view at runs/<ARM>/tb_live/. TRL writes one event
# file per iteration, each restarting at step 0; this stitches them live. The
# post-hoc matplotlib dashboard in section 10 works regardless.
TB_LIVE_LOGGING         = False
TB_SAMPLE_COMPLETIONS_N = 8
PUSH_TO_HUB = False
HUB_ENTITY  = ""          # required when PUSH_TO_HUB is True

# --- Paths --------------------------------------------------------------------
WORKSPACE_ROOT = ""       # "" -> core.runtime.resolve_workspace_root() (section 2)
DATA_ROOT      = ""       # "" -> <workspace>/data ; wins over WORKSPACE_ROOT
# Where this notebook lives inside the Drive mirror, for the Colab chdir in sec. 2.
COLAB_CODE_DIR = "/content/drive/MyDrive/Thesis_PTO_GRPO/Exp4_OpenStack/code/grpo"

# =============================================================================
#  Quick-test overrides -- a VRAM REHEARSAL, not a toy.
#  Every per-forward SHAPE is kept at the real arm's value on purpose: the
#  question the rehearsal answers is "does the real step fit beside the server?",
#  and a step that fits at MAX_COMPLETION_LENGTH=64 / 16-utterance conversations /
#  sub-batch 8 says nothing about one at 200 / 50 / 64. So TRAIN_BATCH_SIZE (the
#  loss-forward width), MAX_COMPLETION_LENGTH, NUM_UTTERANCES_FOR_DATA (the prompt
#  lengths) and LOOKAHEAD_SUB_BATCH_SIZE stay; only the COUNTS shrink. The peak
#  VRAM of each phase lands in iteration_metadata.json (peak_reserved_gib_*).
#  G is dropped to 4 ON PURPOSE: G is encoded in EXPERIMENT_NAME (_G4_), so the
#  rehearsal lands in a DIFFERENT folder than the real arm (Exp4 has no
#  full|quicktest path level), and 16 % 4 == 0 keeps the batch arithmetic legal.
#  Steps: 8 convs x ~20 MCL-eligible slices (50 utterances, MCL 12) ~= 160 prompts,
#  ~140 after the 1-conversation eval split, at (16 x 8) / 4 = 32 prompts/step
#  -> ~4 optimizer steps per iteration (>= 2 needed to see a second step reuse
#  the allocator); SAVE_STEPS=2 so a mid-iteration checkpoint (+ the on_save
#  timing partial and EDA snapshot) is actually written and the resume path can
#  be rehearsed by interrupting iteration 2.
# =============================================================================
if QUICK_TEST:
    NUM_ITERATIONS              = 2
    NUM_CONVERSATIONS_PER_ITER  = 8
    NUM_GENERATIONS             = 4     # -> _G4_ in the arm name -> a disjoint folder
    CONVERSATION_BATCH_SIZE     = 8     # one batch of 8 (the real arm runs 64 at once)
    ORACLE_MAX_CONCURRENCY      = 16
    PATIENT_CONCURRENCY         = 16
    SAVE_STEPS                  = 2
    PUSH_TO_HUB                 = False
    RUN_MODE                    = "quicktest"
    assert TRAIN_BATCH_SIZE % NUM_GENERATIONS == 0, (TRAIN_BATCH_SIZE, NUM_GENERATIONS)
    print("QUICK_TEST: real per-forward shapes, small counts, G=4 -> a separate arm folder")

print("cell 1 read.")
print(f"  method GRPO   K {LOOKAHEAD_K}   MCL {MIN_CONV_LENGTH}   G {NUM_GENERATIONS}   "
      f"rubric {QUESTIONNAIRE_IDS}")
print(f"  oracle  {ORACLE_PROVIDER}:{ORACLE_MODEL_ID}")
print(f"  patient {PATIENT_PROVIDER}:{PATIENT_MODEL_ID}")
print(f"  loop    {NUM_ITERATIONS} iterations x {EPOCHS_PER_ITERATION} epoch(s), seed {SEED}")
print("  EXPERIMENT_NAME is computed in section 6 -- it is deliberately not set here.")

---
## 2. Runtime: host, Drive, workspace root, credentials

Puts `code/` on `sys.path`, resolves the workspace root, and sets up whatever credentials this
particular arm needs.

**The default stack needs no OpenAI key and no Anthropic key** - the patient, the oracle and the
judge all run on the local vLLM server, which authenticates nothing. `authenticate` only hard-fails
on a credential the caller *declares* it needs, and the declaration below is derived from the role
providers in cell 1: flip one role to `"openai"` and the key becomes mandatory, leave the defaults
and nothing vendor-side is required.

Hugging Face is still in the picture because `meta-llama/Llama-3.2-1B` is a gated repo. `HF_TOKEN`
is exported into `os.environ` rather than only handed to `login`, because the vLLM server started in
section 3 is a **subprocess**: it inherits the environment, not the in-process hub session.

The raw `google.colab` Drive mount happens before any project import on purpose - on Colab the code
itself lives inside Drive, so there is nothing to import until the mount lands.

**Off Colab** (a GPU server over SSH) nothing Colab-specific runs: open the notebook from
`code/grpo/` or set `EXP4_WORKSPACE_ROOT`, install `requirements.txt` + `vllm` by hand (cell 0
refuses to install outside Colab), and supply keys as env vars - `get_secret` reads those first.


In [ ]:
import os
import sys


def _find_code_dir(start: str) -> str:
    """Nearest ancestor of *start* that IS, or contains, Exp4's ``code/`` directory.

    Identified structurally (a ``core/`` subdir next to ``naming.py``) rather than by name, so it
    works whether the kernel's cwd is ``code/grpo/``, ``code/`` or the workspace root.
    """
    cur = os.path.abspath(start)
    for _ in range(8):
        for cand in (os.path.join(cur, "code"), cur):
            if (os.path.isdir(os.path.join(cand, "core"))
                    and os.path.isfile(os.path.join(cand, "naming.py"))):
                return os.path.abspath(cand)
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    raise RuntimeError(
        f"could not find Exp4_OpenStack/code/ from {os.path.abspath(start)}. Open this notebook "
        f"from code/grpo/, or os.chdir() there first."
    )


# Colab: mount Drive, then chdir into this notebook's own folder so the walk starts inside the
# tree. Done with the raw google.colab import rather than core.runtime.mount_drive_if_colab
# because on Colab core/ lives inside Drive and does not exist until the mount lands.
#
# OFF Colab (a GPU server over SSH): COLAB_CODE_DIR is never used. Open the notebook from
# code/grpo/ (or set EXP4_WORKSPACE_ROOT to the Exp4_OpenStack dir), install with
# `pip install -r requirements.txt` (the repo root) plus `pip install vllm` -- cell 0 refuses
# to install outside Colab on purpose -- and put the keys in env vars (HF_TOKEN; OPENAI_API_KEY
# / ANTHROPIC_API_KEY only for vendor roles): get_secret reads them FIRST. data/ can be a plain
# directory there rather than a Drive symlink. describe_environment records the card size
# (gpu_total_gib) and the vLLM version into run_metadata.json, so an 80 GB and a 40 GB run
# stay distinguishable.
if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
    if os.path.isdir(COLAB_CODE_DIR):
        os.chdir(COLAB_CODE_DIR)
    else:
        print(f"WARNING: COLAB_CODE_DIR does not exist: {COLAB_CODE_DIR}")

CODE_DIR = _find_code_dir(os.getcwd())
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from core.runtime import (  # noqa: E402  -- must follow the sys.path edit above
    assert_import_order,
    authenticate,
    describe_environment,
    detect_host,
    format_environment,
    resolve_workspace_root,
)

# Inspects sys.modules only -- imports nothing, so calling it cannot create the condition it
# checks for. It turns the sm_120 "torch before trl" segfault into a readable message.
assert_import_order()

HOST = detect_host()
WORKSPACE_ROOT = WORKSPACE_ROOT or resolve_workspace_root()
ENVIRONMENT = describe_environment()
print(format_environment(ENVIRONMENT))
print(f"code dir       : {CODE_DIR}")
print(f"workspace root : {WORKSPACE_ROOT}")

_role_providers = {ORACLE_PROVIDER, PATIENT_PROVIDER, JUDGE_PROVIDER}
AUTH = authenticate(
    hf=True,                                       # Llama-3.2-1B is gated
    openai="openai" in _role_providers,            # only when a role really is API-bound
    anthropic="anthropic" in _role_providers,
)
print(f"credentials    : {AUTH}")
if _role_providers == {"openai_compat"}:
    print("  all three roles are local -- this arm spends $0 in API.")


---
## 3. Start the vLLM server - **before any torch import**

`gpu_memory_utilization` is a **pre-allocation, not a ceiling that grows on demand**: vLLM reserves
that fraction of the card for weights plus KV pool at startup and never gives it back. Two
consequences that decide this cell's position in the notebook:

- the server must be started **first**, so its fixed reservation is carved out before the
  trainer starts claiming the spiky remainder (GRPO's 128-completion generate, the look-ahead
  rollout);
- on the target **A100 80 GB** it is 0.50: `0.50 x 80 = 40 GiB` = E4B's 14.89 GiB of weights +
  a ~22 GiB KV pool, which is what lets 96 patient calls and 128 oracle calls overlap. The
  trainer gets the other ~38 GiB of room against a planned `2.5 + 8.8 + 4.4 + 4.0 = 19.7 GiB`
  envelope (the arithmetic is in cell 1; unmeasured until `QUICK_TEST` writes
  `peak_reserved_gib_*`). On a 40 GB card the same 0.50 leaves the server a ~5 GiB pool and
  the trainer `40 - 20 - 1 = ~19 GiB`: headroom ~= 0, so the fallback starts on the
  16 x 8 batch config, just slower.

`serve_roles` is **idempotent**. A healthy server already listening on the port and serving the
right model is *adopted*, not duplicated - which is what makes re-running this cell safe after a
typo in cell 1, a Colab reconnect, or a kernel restart that did not take the server with it. A port
that answers with a **different** model is a hard error rather than an adoption: silently talking to
the wrong grader would produce a complete, valid-looking, wrongly-scored arm, which is the most
expensive failure available here.

An adopted handle has `process=None` and its `stop()` is a deliberate no-op, so nothing in this
notebook can kill a server it did not start. (If an *adopted* server ever wedges - listening but
not answering - `ensure_alive` raises a dedicated error telling you to kill it by hand.)

`report_weights_gib` reads the real weight figure out of the vLLM startup log. The budget expects
**14.89 GiB** for Gemma-4-E4B-it (9.54 GiB for E2B), read off the HF API - **verify the measured
figure here rather than trusting the arithmetic.**

In [ ]:
from roles import default_serve_util, make_binding, plan_servers
from tools.vllm_serve import report_weights_gib, serve_roles

# One binding per role, built from cell 1. request_timeout is PER ATTEMPT.
_ROLE_SPEC = {
    "oracle":  (ORACLE_PROVIDER,  ORACLE_MODEL_ID,  ORACLE_REQUEST_TIMEOUT,  ORACLE_MAX_RETRIES),
    "patient": (PATIENT_PROVIDER, PATIENT_MODEL_ID, PATIENT_REQUEST_TIMEOUT, PATIENT_MAX_RETRIES),
    "judge":   (JUDGE_PROVIDER,   JUDGE_MODEL_ID,   ORACLE_REQUEST_TIMEOUT,  ORACLE_MAX_RETRIES),
}
_requested = {
    role: make_binding(
        provider, model,
        disable_thinking=DISABLE_THINKING,   # openai_compat only -- vendor APIs 400 on extra keys
        request_timeout=timeout,
        max_retries=retries,
    )
    for role, (provider, model, timeout, retries) in _ROLE_SPEC.items()
}

_SPEC_KW = dict(
    gpu_memory_utilization=VLLM_GPU_MEM_UTIL,
    max_model_len=VLLM_MAX_MODEL_LEN,
    dtype=VLLM_DTYPE,
    extra_args=tuple(VLLM_EXTRA_ARGS),
)

# LOCAL ephemeral disk, never the Drive mount: the server's merged stdout streams through this
# path for the whole arm, and a wedged FUSE write on it stalls the server itself. Identical
# default to the PTO notebook so an adopted handle finds the log (report_weights_gib reads it).
VLLM_LOGS = VLLM_LOG_DIR or ("/content/vllm_logs" if HOST == "colab"
                             else os.path.join(os.getcwd(), "_vllm_logs"))

# The plan is pure data and port assignment is deterministic (sorted by model id), so printing it
# here shows exactly what serve_roles is about to execute -- and the same determinism is what lets
# a resumed session re-plan onto the same ports and adopt what is already running.
_plan = plan_servers(_requested, base_port=VLLM_PORT, **_SPEC_KW)
print(f"server plan: {len(_plan)} process(es) for {len(_requested)} role(s) "
      f"(deduped by model id)")
for _spec in _plan:
    print(f"  {_spec.model}  ->  {_spec.base_url}  "
          f"util {_spec.gpu_memory_utilization}  max_model_len {_spec.max_model_len}  "
          f"dtype {_spec.dtype}")

# The fraction has ONE owner (roles.DEFAULT_SERVE_UTIL); a cell-1 literal that drifted from it
# would pre-allocate a different amount than smoke.py roles / Run_Eval plan for the same grader.
for _spec in _plan:
    assert _spec.gpu_memory_utilization == default_serve_util(_spec.model), (
        f"VLLM_GPU_MEM_UTIL={_spec.gpu_memory_utilization} != roles.default_serve_util"
        f"({_spec.model!r})={default_serve_util(_spec.model)}; fix cell 1 or the table, not one of them"
    )

ROLE_BINDINGS, SERVER_HANDLES = serve_roles(
    _requested,
    base_port=VLLM_PORT,
    log_dir=VLLM_LOGS,
    **_SPEC_KW,
)

# MEASURED, not estimated. The budget expects ~14.89 GiB for E4B (~9.54 for E2B), read off the HF
# API -- check the measured figure against that; a big mismatch means the serving stack is not
# loading what the arithmetic assumed. None means the log was unreadable or vLLM reworded the
# line -- report "unknown" rather than failing on a log-format change, but do not then quote the
# estimate as if it had been checked.
SERVER_WEIGHTS_GIB = {model: report_weights_gib(h) for model, h in SERVER_HANDLES.items()}
for _model, _gib in SERVER_WEIGHTS_GIB.items():
    print(f"  weights: {_model} = {'unknown' if _gib is None else f'{_gib:.2f} GiB'} "
          f"(from the vLLM startup log)")
if not SERVER_HANDLES:
    print("  no openai_compat roles -- nothing was served; every role is a vendor API.")

---
## 4. Oracle sanity gate - run this before spending a GPU-hour

An open-weights grader fails in two ways, and **only one of them is loud**:

1. it ignores the schema or returns the wrong number of item scores. Caught by validation; surfaces
   as retries and then as *biased missingness*, because the calls that fail are correlated with
   conversation difficulty;
2. **it honours the schema perfectly and returns degenerate scores** - every item a 4, near-zero
   variance across conversations. That parses, writes valid parquet, and produces a grader that
   cannot tell any two arms apart. Nothing downstream flags it; the contrast tables just come back
   near zero and look like a finding.

The gate scores a committed fixture of 12 real Exp3 transcripts spanning the quality range, each
carrying the `gpt-4o-mini` scores it actually received. **Hard gates** (block the run): 100%
schema-valid on every requested rubric, and a pooled per-conversation SD that is not degenerate.
**Soft** (reported only): Spearman rank agreement against the reference, and the mean level offset.

`--quick` scores only the two ends of the reference range - it is a pre-flight, and the spread gate
is genuinely weaker at n=2. The **full** report is the Phase 2 gate and should be run once per
grader before any real arm.

The report is archived next to `run_metadata.json` in section 6, not here: the run folder is named
after `EXPERIMENT_NAME`, which has not been computed yet.


In [ ]:
from core.concurrency import run_async
from core.oracle import set_openai_compat_strict
from tools.oracle_sanity import check_gates, format_report, run_sanity, write_report

# Module-level flag, so this also governs the trainer's own oracle calls for the rest of the
# process. Mirrored from the PTO notebook; recorded in run_metadata.json by config_to_metadata.
set_openai_compat_strict(ORACLE_SCHEMA_STRICT)

# Bound EXACTLY as the trainer will bind it -- same model, same endpoint, same provider, same
# max_tokens. Testing a different binding proves nothing about the run: a local model that opens
# with a preamble can fit the JSON at 512 and clip it at 256.
SANITY_REPORT = run_async(run_sanity(
    ROLE_BINDINGS["oracle"],
    questionnaire_ids=QUESTIONNAIRE_IDS,
    # quick=True samples 2 transcripts instead of 12. That existed to save oracle CALLS -- and on
    # the default open stack a call costs nothing, so the saving buys nothing while giving up the
    # only gate that matters: the spread check needs n >= MIN_N_FOR_SPREAD_GATE to tell a
    # template-answering grader apart from two conversations that happened to score alike, and
    # below that it is reported rather than enforced. So: full gate whenever the grader is local,
    # quick only when someone is paying per call.
    quick=(ORACLE_PROVIDER != "openai_compat"),
    concurrency=min(8, ORACLE_MAX_CONCURRENCY),
    max_tokens=ORACLE_MAX_TOKENS,
    max_retries=ORACLE_MAX_RETRIES,
    request_timeout=ORACLE_REQUEST_TIMEOUT,
))
print(format_report(SANITY_REPORT))

SANITY_OK, SANITY_REASONS = check_gates(SANITY_REPORT)
if not SANITY_OK:
    raise RuntimeError(
        "ORACLE UNFIT -- stopping before any GPU time is spent:\n  - "
        + "\n  - ".join(SANITY_REASONS)
        + "\n\nA grader that fails a hard gate does not produce a weaker experiment, it produces "
          "an unmeasurable one. Fix the grader (thinking still on? max_tokens clipping the JSON? "
          "wrong model adopted on the port?) and re-run this cell."
    )
print("\nhard gates PASSED. Note: passing is a floor on being a measuring instrument at all, "
      "not a target -- run the FULL report before committing to an arm.")

---
## 5. Imports: `trl` first, then torch

On the local Blackwell card (sm_120), importing trl *after* torch has initialised CUDA **segfaults**
- exit 139, no Python traceback, no OOM, nothing to catch. It is a native initialisation-order
conflict, not a bug in the trainers, and Colab is unaffected (which is why the full runs run there).
Everything below `trl` pulls torch in transitively: transformers, peft, datasets, `core.policy`,
`core.lookahead`, and therefore `core.config`'s builders.

**A bare `import trl` is not enough.** trl 1.4.0's top level is a `_LazyModule`: after
`import trl`, `trl.trainer` is not in `sys.modules` and `GRPOTrainer` has not been materialised, so
none of the native initialisation has happened yet. Measured locally, 3/3 reproductions:
`import trl` -> `torch.cuda.is_available()` -> `from trl import GRPOTrainer` segfaults, while
`from trl import GRPOTrainer` -> `torch.cuda.is_available()` does not. The concrete symbol import
below is therefore what actually satisfies the ordering; `core.runtime.assert_import_order` checks
`"trl" in sys.modules`, which the lazy shell already satisfies, so it cannot see this case.

`grpo_trainer` is imported **qualified**. It defines `build_grpo_config` and `write_run_metadata`,
which share their names with functions in `core.config` that do different jobs - `core.config`
builds the frozen config *bundle* from cell-1 globals and serialises a prepared payload, while
`grpo_trainer`'s build a `trl.GRPOConfig` and compose that payload. Star-importing both into one
namespace is how those get silently swapped.


In [ ]:
# trl FIRST -- these two imports exist for their effect on native init order, not for their names.
# The CONCRETE import is the one that matters: trl 1.4.0's top level is a _LazyModule, so after a
# bare `import trl` nothing of trl.trainer has actually loaded. Measured on the local sm_120 card
# (3/3): bare import -> torch.cuda.is_available() -> `from trl import GRPOTrainer` segfaults at
# exit 139, and pulling the symbol first does not.
import trl  # noqa: F401
from trl import GRPOConfig, GRPOTrainer  # noqa: F401

import gc  # noqa: E402
import dataclasses  # noqa: E402
import random  # noqa: E402

# datasets BEFORE torch -- the second native-init pair. MEASURED on the local sm_120 card:
# importing torch first makes the datasets import an access violation (exit 139) inside
# pyarrow.dataset. Harmless-looking line order, unrecoverable crash. See CLAUDE.md.
import datasets  # noqa: E402

import torch  # noqa: E402
import peft  # noqa: E402
import transformers  # noqa: E402

from core.concurrency import AsyncPrimitives  # noqa: E402
from core.config import build_grpo_config, validate_config  # noqa: E402
from core.policy import (  # noqa: E402
    compute_cumulative_step_offset,
    list_iteration_checkpoints,
    patch_generate,
    resolve_start_state,
    setup_base_model,
    setup_tokenizer,
    sync_pad_token,
    vram_report,
)
from core.tb import RunTBLogger, plot_iteration_metrics, scan_scalar_tags  # noqa: E402
from roles import make_client  # noqa: E402
from system_prompts_builder import generate_all_permutations  # noqa: E402
from tools.generate_convs import therapist_prompt_pair  # noqa: E402

# QUALIFIED on purpose -- two of its names collide with core.config's.
from grpo import grpo_trainer  # noqa: E402

print(f"torch {torch.__version__}  |  transformers {transformers.__version__}  |  "
      f"trl {trl.__version__}  |  peft {peft.__version__}  |  datasets {datasets.__version__}")
print(f"cuda available: {torch.cuda.is_available()}  "
      f"({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no device'})")

---
## 6. Freeze the config bundle - and compute `EXPERIMENT_NAME`

`build_grpo_config(globals())` reads only recognised ALL-CAPS names, freezes them into typed
dataclasses, **computes** the arm name from the very values it is about to freeze, derives every
path from that name, and validates the whole bundle. It also warns about ALL-CAPS globals that were
never read but look like a typo of one that is - because a misspelled knob (`LOOKAHED_K = 5`) is
otherwise completely silent: the default is used, the name says `LA0`, and the run is a different
experiment than the one that was configured.

```
{GRPO|PTO}4_{QTAG}_LA{K}_MCL{N}_{G{G} | M{M}_PT{mode}}_O{otag}_Pat{ptag}_Th{ttag}
```

`validate_config` is called again below. That is a deliberate, visible re-run of the gate the
builder already ran, so the cell reads as "this bundle was checked" without opening the module.

Then `run_metadata.json` is written - **plus a line appended to `run_metadata_history.jsonl`**.
Exp3 overwrote its metadata in place, so a resume under changed knobs restamped the whole arm,
earlier iterations included, and the values those iterations actually ran under were simply gone.
The current file is still overwritten (a reader wants exactly one "what is this arm configured
as?"), but nothing is lost.

The section-4 sanity report is archived here, next to that metadata, so it travels with the run.


In [ ]:
train_cfg, roles_cfg, gen_cfg, oracle_cfg, la_cfg, paths = build_grpo_config(globals())

# For later cells to print. It is never read back into the config -- the builder ignores any
# EXPERIMENT_NAME in globals() and warns if a stale one disagrees with what it computed.
EXPERIMENT_NAME = train_cfg.experiment_name

# The same gate build_grpo_config already ran, re-run where a reader can see it.
validate_config(train_cfg, roles_cfg, gen_cfg, oracle_cfg, la_cfg, paths)

paths.ensure_run_dir()
SANITY_REPORT_PATH = write_report(SANITY_REPORT, paths.run_dir)

RUN_METADATA_PATH = grpo_trainer.write_run_metadata(
    train_cfg, roles_cfg, gen_cfg, oracle_cfg, la_cfg, paths,
    extra={
        "host": HOST,
        "environment": ENVIRONMENT,
        "credentials_resolved": AUTH,
        "serve_specs": [dataclasses.asdict(h.spec) for h in SERVER_HANDLES.values()],
        "server_weights_gib": SERVER_WEIGHTS_GIB,
        "oracle_sanity": {
            "passed": SANITY_OK,
            "quick": SANITY_REPORT.quick,
            "model": SANITY_REPORT.model,
            "base_url": SANITY_REPORT.base_url,
            "n_items": SANITY_REPORT.n_items,
            "report_path": SANITY_REPORT_PATH,
        },
    },
)

print()
print("=" * 78)
print(f"EXPERIMENT_NAME (computed): {EXPERIMENT_NAME}")
print("=" * 78)
print(f"  prompts/step   {train_cfg.prompts_per_step}  "
      f"({train_cfg.generation_batch_size} completions / G={train_cfg.num_generations})")
print(f"  run dir        {paths.run_dir}")
print(f"  conversations  {paths.conv_root}")
print(f"  oracle sanity  {SANITY_REPORT_PATH}")
print(f"  run metadata   {RUN_METADATA_PATH}")
print(f"  history log    {paths.run_metadata_history_path}")


---
## 7. Tokenizer, personas, base policy, resume state

**No LoRA is attached here, on purpose.** `grpo_trainer.ensure_peft_policy` attaches it inside each
iteration, *after* the generation phase, for two reasons:

- iteration 1 must generate `model_iter_0` with the genuinely untrained base policy. Attaching
  first would be numerically identical (a fresh LoRA is identity at step 0) but would make the
  "base" folder a claim rather than a fact;
- the reward closure captures the policy object so that look-ahead rolls out under the weights
  being trained. If `GRPOTrainer(peft_config=...)` did the wrapping instead, the closure would hold
  the *unwrapped base* and every simulated therapist turn of that iteration would come from a
  policy with no adapter - silently, since the divergence only grows as training proceeds.

`resolve_start_state` decides where to start from what is on disk: a fresh start, a mid-iteration
crash (resume from the latest **valid** HF checkpoint, walking back over a torn write), or a clean
boundary between iterations. `resume_checkpoint` applies to the **first iteration of this process
only**.

One `AsyncPrimitives` instance for the whole process, and it must be *this* one: TRL awaits the
reward closure on its own event loop, and an asyncio semaphore cached anywhere else raises
"attached to a different loop" partway into training.

One client serves both the patient and the oracle - in the default stack they are the same endpoint.
It is built from the **oracle** binding, the one with the longer per-attempt timeout, so that each
role's own `asyncio.wait_for` is the effective bound rather than a socket configured for the other
role.


In [ ]:
tokenizer = setup_tokenizer(train_cfg.tokenizer_id)
print(f"tokenizer: {train_cfg.tokenizer_id}  (vocab {len(tokenizer)}, pad {tokenizer.pad_token!r})")

base_policy = setup_base_model(train_cfg.base_model_id, use_4bit=train_cfg.use_4bit)
sync_pad_token(base_policy, tokenizer)
patch_generate(base_policy, tokenizer)

# generate_all_permutations draws the therapist's NAME from global random (random.choice inside
# choose_random_therapist_name), so the seed is re-applied immediately before the call -- a
# resumed process must rebuild the SAME therapist system prompt, or the arm's second half runs
# under a different prompt than its first. Mirrors the PTO notebook exactly.
random.seed(train_cfg.seed)
# The FULL permutation list in canonical order -- it is indexed by persona id downstream, so a
# shuffled subset here would silently pair each conversation with the wrong patient.
permutations = generate_all_permutations(only_expert_therapist=True)
therapist_system_prompt, therapist_init_utterance = therapist_prompt_pair(permutations)
if gen_cfg.num_conversations_per_iter > len(permutations):
    raise ValueError(
        f"NUM_CONVERSATIONS_PER_ITER={gen_cfg.num_conversations_per_iter} exceeds the "
        f"{len(permutations)} available personas"
    )
print(f"personas: {len(permutations)} (running {gen_cfg.num_conversations_per_iter}/iteration)")

# Conversation files are named by the STABLE persona id (pers07.csv is persona 7 in every
# iteration, forever). Exp3 named them by the shuffled processing index, so every EDA module had
# to replay Random(seed + k + 1) to pair anything. There is nothing to replay here.
_existing = list_iteration_checkpoints(paths.run_dir)
print(f"iterations already on disk: "
      f"{[f'iteration_{n}' for n, _ in _existing] if _existing else '(none)'}")

start_iteration, policy, resume_checkpoint = resolve_start_state(
    paths.run_dir, base_policy, tokenizer
)
cumulative_step_offset = compute_cumulative_step_offset(paths.run_dir)
print(f"starting at iteration {start_iteration} of {train_cfg.num_iterations}   "
      f"(cumulative step offset {cumulative_step_offset})")

primitives = AsyncPrimitives(
    oracle_concurrency=oracle_cfg.max_concurrency,
    patient_concurrency=gen_cfg.patient_concurrency,
)


def client_factory():
    """A fresh client for the shared endpoint.

    ONE binding serves the oracle AND every patient call, so it is built from the oracle binding
    and the two roles must resolve to the same endpoint -- which validate_config enforced above
    (core.config._roles_errors); a split stack cannot be addressed from here.

    Note the async entry points re-resolve their client per event loop anyway (make_client is
    loop-keyed) -- this object mostly serves as the plumbed-through default. Called after a vLLM
    restart, which invalidates every cached client.
    """
    return make_client(roles_cfg.oracle)


client = client_factory()
tb_logger = RunTBLogger(paths.run_dir, enabled=train_cfg.tb_live_logging)
print(f"policy resident: {vram_report()['reserved_gib']:.1f} GiB reserved by this process")

---
## 8. The orchestration loop

The loop is **here**, visible, not behind a convenience wrapper. Exp3 had a `run_iterative_training`
that duplicated its notebook loop; the two drifted, and calling the wrapper silently ran a different
experiment. When a run stalls at 3 a.m. on Colab, the thing that makes it diagnosable is being able
to read what it is about to do.

Each pass calls `grpo_trainer.run_one_iteration`, which does generate -> extract -> split -> train ->
save and calls `ensure_alive` at its own generate and train phase boundaries. The probe at the top
of the loop body is the third boundary - between iterations - and it is spelled out here so the
server-health story is readable without opening the module.

**Resume happens in three places** and the third is the one that is easy to forget: conversations
already on disk are reloaded rather than regenerated; the HF trainer resumes from `resume_checkpoint`;
and the EDA recorder is reloaded from the snapshot stored *inside* that checkpoint. HuggingFace
resumes by fast-forwarding through batches it already consumed, and **a fast-forwarded batch never
re-invokes the reward function** - so without that reload the iteration's `generations.jsonl` would
silently contain only the rows produced after the resume point: a file that looks complete and is
missing its first half.

Each iteration builds a **fresh** trainer, so Adam's moments reset while the LoRA weights carry over
- a warm restart per iteration. That is deliberate, and it is matched on the PTO side.

The loop stops itself rather than continuing on a bad state: the reward function raises when the
oracle success rate falls below `min_success_ratio` (training on a biased subset is worse than
stopping), and `ensure_alive` raises when a vLLM server exhausts its restart budget (a server that
keeps dying is a configuration problem, almost always `gpu_memory_utilization` colliding with the
trainer's peak). Both mean "fix the cause, then re-run" - the iteration resumes from its last
checkpoint.


In [ ]:
iterations_run = []

if start_iteration > train_cfg.num_iterations:
    print(f"nothing to train: iterations 1..{train_cfg.num_iterations} are already complete. "
          f"Run section 9 to refill the final adapter's eval conversations if needed.")

for iteration in range(start_iteration, train_cfg.num_iterations + 1):
    # Phase boundary. run_one_iteration repeats this at its own generate/train boundaries; this
    # one catches a server that died BETWEEN iterations, and costs one HTTP probe when healthy.
    fresh_client = grpo_trainer.ensure_servers_alive(
        SERVER_HANDLES, client_factory=client_factory, phase=f"iteration {iteration}"
    )
    if fresh_client is not None:
        client = fresh_client

    result = grpo_trainer.run_one_iteration(
        iteration=iteration,
        policy=policy,
        tokenizer=tokenizer,
        client=client,
        cfg=train_cfg,
        gen=gen_cfg,
        roles=roles_cfg,
        oracle_cfg=oracle_cfg,
        la_cfg=la_cfg,
        paths=paths,
        primitives=primitives,
        permutations=permutations,
        therapist_system_prompt=therapist_system_prompt,
        therapist_init_utterance=therapist_init_utterance,
        start_iteration=start_iteration,
        resume_checkpoint=resume_checkpoint,
        cumulative_step_offset=cumulative_step_offset,
        tb_logger=tb_logger,
        server_handles=SERVER_HANDLES,
        client_factory=client_factory,
    )

    # run_one_iteration returns the TRAINED policy -- rebinding it is not optional, or the next
    # iteration would generate with the previous weights.
    policy = result["policy"]
    client = result.get("client", client)
    cumulative_step_offset += int(result.get("step_delta", 0))
    # Consumed. It applies to the FIRST iteration of this process only; a fresh resume means
    # re-running section 7.
    resume_checkpoint = None
    iterations_run.append(iteration)

    gc.collect()
    torch.cuda.empty_cache()

print(f"\nthis process trained {len(iterations_run)} iteration(s): {iterations_run or '(none)'}")


---
## 9. Post-loop eval pass: `model_iter_{NUM_ITERATIONS}`

Every trained iteration gets its eval conversations for free - iteration `n` generates with the
iter-(`n`-1) policy, so those conversations *are* the eval set for model state `n-1`. The **last**
adapter never generates anything under that rule, so `N` iterations would otherwise produce `N`
conversation folders and the final policy would have no measurement at all. This pass closes the
gap, which is why `N` iterations yield `N+1` folders.

No prompts are extracted here - nothing is going to be trained on them. It is **idempotent**:
personas whose CSVs already exist are skipped, so re-running after a successful pass costs a
directory listing. The timing is logged against `iteration_{NUM_ITERATIONS}` under its own
`eval_gen_s` phase key, precisely so a cost analysis can exclude it from training cost without
losing it.


In [ ]:
final_eval = grpo_trainer.run_final_eval(
    policy=policy,
    tokenizer=tokenizer,
    client=client,
    cfg=train_cfg,
    gen=gen_cfg,
    roles=roles_cfg,
    paths=paths,
    primitives=primitives,
    permutations=permutations,
    therapist_system_prompt=therapist_system_prompt,
    therapist_init_utterance=therapist_init_utterance,
    server_handles=SERVER_HANDLES,
    client_factory=client_factory,
)
client = final_eval.get("client", client)
tb_logger.close()

_saved = [n for n, _ in list_iteration_checkpoints(paths.run_dir)]
print("\n" + "=" * 78)
print("RUN COMPLETE")
print("=" * 78)
print(f"  arm            {EXPERIMENT_NAME}")
print(f"  iterations     {_saved or '(none)'}  ({len(iterations_run)} trained in this process)")
print(f"  effective      {train_cfg.total_effective_epochs:g} epochs "
      f"({train_cfg.num_iterations} x {train_cfg.epochs_per_iteration})")
print(f"  run dir        {paths.run_dir}")
print(f"  conversations  {paths.conv_root}")
print(f"    expected     model_iter_0 .. model_iter_{train_cfg.num_iterations}")
print(f"  final eval     {final_eval['conv_dir']}  "
      f"({final_eval['n_conversations']}/{final_eval['n_requested']} conversations)")
print("=" * 78)

# The vLLM server is deliberately LEFT RUNNING: the eval scoring pass wants the same endpoint, and
# an adopted handle's stop() is a no-op by design anyway. Uncomment to free its pre-allocation
# (VLLM_GPU_MEM_UTIL x the card, 40 GiB on the 80 GB A100) now.
# for _h in SERVER_HANDLES.values():
#     _h.stop()

---
## 10. Inspection: TensorBoard and the cross-iteration dashboard

`scan_scalar_tags` answers "what did this run actually log?" cheaply (tag names only), which is the
right first question when something looks wrong.

TRL writes **one event file per iteration, each starting at `global_step` 0**, so the TB web UI
shows `N` disconnected curves. Two views fix that from opposite ends: the opt-in `tb_live/` run
(`TB_LIVE_LOGGING`) logs at the cumulative step *during* training, and `plot_iteration_metrics`
stitches the per-iteration files *after the fact*, applying per-iteration step offsets so the curves
chain end to end with dotted vlines at the boundaries.


In [ ]:
import socket
from pathlib import Path

scan_scalar_tags(paths.run_dir)

# Bind port 0 to get a free one -- a fixed port collides across notebook sessions and the failure
# looks like TensorBoard being broken.
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as _s:
    _s.bind(("127.0.0.1", 0))
    tb_port = _s.getsockname()[1]

# Forward slashes: the IPython magic re-parses Windows backslashes as escape sequences, which
# silently turns the path into garbage (a \U... sequence gets eaten outright).
tb_logdir = Path(paths.run_dir).as_posix()

print(f"\nTensorBoard on port {tb_port}")
print(f"  logdir: {tb_logdir}")
print("  In the run selector, 'tb_live' (if enabled) is the continuous, smoothable cross-iteration")
print("  view; the per-iteration 'iteration_N/training/tb_logs' runs each restart at step 0.")
print(f"  If the inline iframe does not render (common in VS Code), open http://localhost:{tb_port}")
%reload_ext tensorboard
%tensorboard --logdir "{tb_logdir}" --port {tb_port}


In [ ]:
%matplotlib inline

# Cross-iteration dashboard, read from the per-iteration event files. method="grpo" pins the pane
# set rather than letting it be auto-detected from whichever tags happen to be present -- useful on
# a partially-trained arm, where auto-detection has less to go on. Degrades rather than raising: a
# missing tag drops its series, and an arm with no event files returns None.
plot_iteration_metrics(paths.run_dir, method="grpo")


---
## What next

1. **Score.** Run `eda/notebooks/scoring/Run_Eval.ipynb`. Arms are auto-discovered from disk, so
   there is no registry to edit when this run appears; scores land in
   `data/eval_scores/judge=<tag>/rep=<r>/metric=<M>/<EXPERIMENT_NAME>/model_iter_<N>.parquet`, one
   file per model state rather than Exp3's ~50k single-row CSVs.
2. **Analyse.** From `Exp4_OpenStack/eda/`, run `_selfcheck --fast` and then `render_results.py`.

## Resuming

Re-run this notebook top-to-bottom. Nothing needs a flag flipped:

- **section 3** adopts the vLLM server that is already listening instead of duplicating it;
- **section 7** reads what is on disk - `iteration_N/adapter/` holding its adapter FILES is the
  definition of "that iteration is done" (a torn save is treated as incomplete), so completed
  iterations are skipped and the loop range starts after them;
- a **crashed** iteration resumes from its latest *valid* HF checkpoint (`get_latest_valid_hf_checkpoint`
  walks back over a torn write - which is why `SAVE_TOTAL_LIMIT >= 2` matters), reloading the EDA
  recorder from the snapshot inside it. The policy handed to the resumed trainer carries the
  **iteration-start** weights - TRL snapshots what it is handed as the KL reference - and
  `grpo_trainer.restore_default_adapter` then loads the checkpoint's **trained** `default`
  weights before `train()`. That explicit load is load-bearing: every checkpoint holds `default`
  at its root and TRL's `ref` copy in a `ref/` subdirectory, and transformers' own resume load
  reads *only* the subdirectories - without it a "resumed" iteration silently continued from
  its starting weights while the optimizer state and step counter said otherwise. So a resumed
  iteration keeps the documented reference AND the trained weights.
  Its generation phase is **reload-only**: the dataset must match the crashed process's exactly
  (HF fast-forwards batches positionally), so nothing is regenerated on that path;
- conversations already written are **reloaded, not regenerated**, so a killed generation phase is
  cheap to restart. **Each phase logs its own timing line when it completes**, so a preempted
  process still leaves its finished phases on the cost record.

Two things a resume does *not* do: it does not reload a finished adapter into an in-memory policy
that was handed to `run_one_iteration` with `skip_if_complete` (the loop range makes that path
unreachable in normal use), and it does not undo a knob you changed between processes. The changed
value is recorded - `run_metadata.json` is overwritten but every process appends its own line to
`run_metadata_history.jsonl` first, so an arm that ran under two configurations says so.

**If the arm looks unfinished, check the cloud before regenerating.** `data/` is a Google Drive
symlink and the mount can wedge on a single folder: in Exp3 a fully populated `model_iter_N/` read
as 0 files with an intermittent `WinError 1450` while every conversation was present in Drive the
whole time. A Drive restart fixed it; the alternative was a needless regeneration pass.